# sliding ship

add sliding along buildings instead of pure stopping

In [ ]:
from pathlib import Path
import hashlib
import sys

import matplotlib.pyplot as plt
from matplotlib.patches import Polygon as PolygonPatch
from netCDF4 import Dataset
import numpy as np
import pandas as pd
from shapely import wkt
from shapely.geometry import box
from shapely.ops import transform, unary_union
from IPython.display import display

HERE = Path.cwd()
if not (HERE / 'setrun.py').exists(): HERE = HERE / 'examples' / 'seattle_debris'
if not (HERE / 'setrun.py').exists(): raise FileNotFoundError('Run this notebook from the Seattle example or repository root')
sys.path.insert(0, str(HERE.parents[1] / 'src'))
from geoclaw_debris.ship_tracking import Flow, Ship, hull_polygon, simulate

FLOW_FILE = HERE / 'pike_place_surge12_fgout.nc'
TERRAIN_FILE = HERE / 'seattle_buildings_mhw.tt3'
INVENTORY_FILE = HERE.parents[1] / 'SeattleBuildings' / 'Seattle_Inventory.csv'

CASES = [
    dict(name='Pinch point', start=(-122.348685, 47.610661), heading_deg=0.,
         length_m=30., beam_m=8., draft_m=1.5, start_time=50., end_time=180., dt=2.),
    dict(name='Temporary block', start=(-122.347722063, 47.610498408), heading_deg=58.6,
         length_m=30., beam_m=8., draft_m=1.5, start_time=50., end_time=240., dt=2.),
    dict(name='Southwest launch', start=(-122.3479215737869, 47.61040846599875), heading_deg=45.,
         length_m=30., beam_m=8., draft_m=1.5, start_time=0., end_time=180., dt=2.),
]

In [ ]:
terrain_hash = hashlib.sha256(TERRAIN_FILE.read_bytes()).hexdigest()
with Dataset(FLOW_FILE) as ds:
    assert ds.terrain_sha256 == terrain_hash, 'flow terrain is stale: you must remake it'
    flow = Flow(ds)
    domain = box(flow.lon.min(), flow.lat.min(), flow.lon.max(), flow.lat.max())
    footprints = [shape for shape in map(wkt.loads, pd.read_csv(INVENTORY_FILE).geometry)
                  if shape.intersects(domain)]
    buildings = unary_union(footprints)
    metric_footprints = [transform(flow.to_xy, shape) for shape in footprints]

    results = []
    for case in CASES:
        ship = Ship(length_m=case['length_m'], beam_m=case['beam_m'], draft_m=case['draft_m'])
        options = {key: case[key] for key in ('heading_deg', 'start_time', 'end_time', 'dt')}
        print('running: ', case['name'], flush=True)
        stopped = simulate(flow, buildings, ship, case['start'], **options)
        sliding = simulate(flow, buildings, ship, case['start'], contact_mode='slide', **options)
        results.append(dict(case=case, ship=ship, stopped=stopped, sliding=sliding))
print('completed: ', len(results), ' cases')

In [ ]:
def first_time(path, status): return next((row['time_s'] for row in path if row['status'] == status), np.nan)

def longest_block_seconds(path, dt):
    longest = run = 0
    for row in path:
        run = run + 1 if row['status'] == 'blocked' else 0
        longest = max(longest, run)
    return longest*dt

summary = []
for result in results:
    case, stopped, sliding = result['case'], result['stopped'], result['sliding']
    start = np.array([sliding[0]['x_m'], sliding[0]['y_m']])
    finish = np.array([sliding[-1]['x_m'], sliding[-1]['y_m']])
    summary.append(dict(case=case['name'], first_contact_s=first_time(stopped, 'building contact'),
                        first_grounding_s=first_time(sliding, 'grounding'),
                        longest_block_s=longest_block_seconds(sliding, case['dt']),
                        final_status=sliding[-1]['status'], net_drift_m=round(np.linalg.norm(finish-start), 1)))
display(pd.DataFrame(summary))

In [ ]:
def draw_hull(ax, row, ship, origin, color, label, alpha=.8):
    polygon = hull_polygon((row['x_m'], row['y_m']), np.deg2rad(row['heading_deg']), ship)
    ax.add_patch(PolygonPatch(np.asarray(polygon.exterior.coords)-origin, facecolor=color,
                              edgecolor='#222', lw=1.2, alpha=alpha, label=label, zorder=4))

fig, axes = plt.subplots(1, len(results), figsize=(14, 6), constrained_layout=True)
for ax, result in zip(np.atleast_1d(axes), results):
    case, ship = result['case'], result['ship']
    stopped, sliding = result['stopped'], result['sliding']
    origin = np.array([sliding[0]['x_m'], sliding[0]['y_m']])
    view = box(origin[0]-75, origin[1]-65, origin[0]+95, origin[1]+75)
    for footprint in metric_footprints:
        if not footprint.intersects(view): continue
        pieces = footprint.geoms if hasattr(footprint, 'geoms') else [footprint]
        for piece in pieces:
            if piece.geom_type == 'Polygon':
                ax.add_patch(PolygonPatch(np.asarray(piece.exterior.coords)-origin, facecolor='#626873', edgecolor='#333', lw=.6, zorder=1))
    first_contact = next((row for row in stopped if row['status'] == 'building contact'), None)
    draw_hull(ax, sliding[0], ship, origin, 'gold', 'launch')
    if first_contact: draw_hull(ax, first_contact, ship, origin, 'tomato', 'first contact', .6)
    xy = np.array([(row['x_m'], row['y_m']) for row in sliding]) - origin
    ax.plot(xy[:, 0], xy[:, 1], color='crimson', lw=2, label='sliding path', zorder=3)
    draw_hull(ax, sliding[-1], ship, origin, 'mediumorchid', 'final hull', .7)
    ax.set(xlim=(-75, 95), ylim=(-65, 75), xlabel='east of launch (m)', ylabel='north of launch (m)', title=case['name'])
    ax.set_aspect('equal')
    ax.legend(loc='lower right', fontsize=8)
plt.show()

In [ ]:
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

CASE_INDEX = 2
result = results[CASE_INDEX]
case, ship, path = result['case'], result['ship'], result['sliding']
origin = np.array([path[0]['x_m'], path[0]['y_m']])
view = box(origin[0]-75, origin[1]-65, origin[0]+95, origin[1]+75)
ix = abs(flow.lon-case['start'][0]) < .0014
iy = abs(flow.lat-case['start'][1]) < .0010
lon, lat = np.meshgrid(flow.lon[ix], flow.lat[iy], indexing='ij')
x, y = flow.to_xy(lon, lat)
with Dataset(FLOW_FILE) as ds:
    n = int(np.argmin(abs(ds['time'][:] - path[0]['time_s'])))
    depth = np.asarray(ds['h'][n])[np.ix_(ix, iy)]

fig, ax = plt.subplots(figsize=(8, 6))
water = ax.pcolormesh(x-origin[0], y-origin[1], np.ma.masked_where(depth < .1, depth), cmap='Blues', vmin=0, vmax=12, shading='auto')
fig.colorbar(water, ax=ax, label='depth at launch (m)', shrink=.8)
for footprint in metric_footprints:
    if not footprint.intersects(view): continue
    pieces = footprint.geoms if hasattr(footprint, 'geoms') else [footprint]
    for piece in pieces:
        if piece.geom_type == 'Polygon': ax.add_patch(PolygonPatch(np.asarray(piece.exterior.coords)-origin, facecolor='#626873', edgecolor='#333', lw=.6))
outline = hull_polygon((path[0]['x_m'], path[0]['y_m']), np.deg2rad(path[0]['heading_deg']), ship)
hull = PolygonPatch(np.asarray(outline.exterior.coords)-origin, facecolor='tomato', edgecolor='red', lw=2)
ax.add_patch(hull)
trail, = ax.plot([], [], color='crimson', lw=2)
ax.set(xlim=(-75, 95), ylim=(-65, 75), xlabel='east of launch (m)', ylabel='north of launch (m)')
ax.set_aspect('equal')
fig.tight_layout()

def update(i):
    row = path[i]
    outline = hull_polygon((row['x_m'], row['y_m']), np.deg2rad(row['heading_deg']), ship)
    hull.set_xy(np.asarray(outline.exterior.coords)-origin)
    trail.set_data([r['x_m']-origin[0] for r in path[:i+1]], [r['y_m']-origin[1] for r in path[:i+1]])
    ax.set_title(f"{case['name']}: {row['time_s']:.0f} s, {row['status']}")

anim = FuncAnimation(fig, update, frames=range(0, len(path), 2), interval=120)
plt.close(fig)
HTML(anim.to_jshtml())